In [1]:
print('check check')

check check


In [2]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
INTERIM, SPLITS = ROOT / "data/interim", ROOT / "data/splits"

counts = (pd.read_parquet(SPLITS / "train_split.parquet", columns=["work_id"])
            .work_id.value_counts().head(200))

books = pd.read_parquet(
    INTERIM / "english_works_all.parquet",
    columns=["work_id", "title", "year_first", "pages_median",
             "rating_weighted", "ratings_total", "genres"],
).set_index("work_id")

rows = []
for wid in counts.index:
    if wid not in books.index:
        continue
    b = books.loc[wid]
    if hasattr(b, "iloc") and b.ndim > 1:
        b = b.iloc[0]
    rows.append({
        "id": str(wid),
        "title": b.title,
        "year": None if pd.isna(b.year_first) else int(b.year_first),
        "pages": None if pd.isna(b.pages_median) else int(b.pages_median),
        "rating": None if pd.isna(b.rating_weighted) else round(float(b.rating_weighted), 2),
        "ratings": int(b.ratings_total),
        "genres": list(b.genres) if b.genres is not None else [],
    })

out = ROOT / "docs" / "data"
out.mkdir(parents=True, exist_ok=True)
(out / "popular.json").write_text(json.dumps({"works": rows}, ensure_ascii=False))
print(f"{len(rows)} works written")

200 works written


In [3]:
import json
d = json.load(open(ROOT / "docs/data/popular.json"))
print(len(d["works"]))
d["works"][:3]

200


[{'id': '4640799',
  'title': "Harry Potter and the Sorcerer's Stone (Harry Potter, #1)",
  'year': 1997,
  'pages': 285,
  'rating': 4.45,
  'ratings': 4970387,
  'genres': ['fantasy_paranormal']},
 {'id': '2792775',
  'title': 'The Hunger Games (The Hunger Games, #1)',
  'year': 2008,
  'pages': 384,
  'rating': 4.34,
  'ratings': 5064668,
  'genres': ['young_adult']},
 {'id': '3212258',
  'title': 'Twilight (Twilight, #1)',
  'year': 2005,
  'pages': 476,
  'rating': 3.57,
  'ratings': 3991256,
  'genres': ['fantasy_paranormal']}]